# 01 — Methods figures

This notebook builds the figures and parameter tables for the **Methods**
section. Nothing here is a heavy computation — it draws schematics, a map,
a timeline, and prints the fixed inputs.

**How each section works:** a code cell imports a generator from
`scripts/figures/`, calls it (which writes a PDF into
`paper/letter/figures/`), and then `show_fig(...)` displays that PDF
inline so you can see it immediately. Run the cells top to bottom.

| Output | Generator |
|---|---|
| Fig 1 — Probe schematic + published K(z) | `scripts/figures/make_intro_figures.py` |
| Fig 2 — Apollo landing-site context map | `scripts/figures/make_context_map_figure.py` |
| Fig 3 — Apollo HFE meter-scale-sensor timeline | `scripts/figures/make_apollo_timeline.py` |
| Fig 4 — Diurnal amplitude vs depth (borestem diagnostic) | `scripts/figures/make_letter_figures.py::fig_amplitude_vs_depth` |
| Table 1 — Fixed solver inputs | inlined below |

All figures overwrite the shipped PDFs in `paper/letter/figures/`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts' / 'figures'))
print('Project root:', ROOT)
FIGS = ROOT / "paper" / "letter" / "figures"

from IPython.display import Image, display
import io
def show_fig(name):
    """Display a figure inline. Prefers the PNG companion; falls back to
    rendering the PDF on the fly via pymupdf if no PNG exists."""
    stem = name.replace('.pdf','').replace('.png','')
    png = FIGS / f"{stem}.png"
    if png.exists():
        display(Image(filename=str(png))); return
    pdf = FIGS / f"{stem}.pdf"
    if pdf.exists():
        try:
            import fitz
            pix = fitz.open(str(pdf))[0].get_pixmap(dpi=140)
            display(Image(data=pix.tobytes("png")))
        except Exception as e:
            print(f"  (couldn't render {pdf.name}: {e})")
    else:
        print(f"(no figure file for {stem})")


## Fig 1 — Probe schematic + published K(z)

Two panels orienting the reader: a schematic of how the HFE probe sits in
the drilled borehole (which sensor is at which depth), and the Hayne
(2017) thermal-conductivity profile K(z), which rises steeply from a very
low surface value to its deep value over the e-folding depth H.

**What to look for:** how shallow the steep part of K(z) is — almost all
the variation happens in the top ~10 cm, which is why a fine near-surface
grid matters.

In [ ]:
import make_intro_figures
make_intro_figures.fig_intro_probe()
show_fig('fig_intro_probe')

## Fig 2 — Apollo landing-site context map

A lunar near-side map marking the two boreholes: **Apollo 15** at Hadley
Rille (26°N) and **Apollo 17** at Taurus–Littrow (20°N). Context for why
the two sites have different regolith and slightly different illumination.

In [ ]:
import make_context_map_figure
make_context_map_figure.main()
show_fig('fig_context_map')

## Fig 3 — Apollo HFE meter-scale-sensor timeline

The 1971–1977 temperature record for each deep sensor. Early on the
sensors are still equilibrating after emplacement; later they settle into
a steady seasonal cycle. We fit only the **stable window** (the settled
portion), and this figure shows how that window is chosen per sensor.

**What to look for:** the flattening of each trace — the retrieval uses
the late, stable part, not the noisy startup transient.

In [ ]:
import make_apollo_timeline
make_apollo_timeline.main()
show_fig('fig_apollo_timeline')

## Fig 4 — Diurnal amplitude vs depth (the borestem diagnostic)

The size of the day–night temperature swing as a function of depth. It
should decay smoothly with depth (heat diffusion damps the surface wave).
Above ~80 cm the metal borestem short-circuits heat and **distorts** that
clean decay.

**What to look for:** the kink/excess amplitude in the shallowest sensors
— that distortion is exactly why we discard everything above the 80 cm
borestem cut.

In [ ]:
from make_letter_figures import fig_amplitude_vs_depth
fig_amplitude_vs_depth()
show_fig('fig_amplitude_vs_depth')

## Table 1 — Fixed inputs to the 1-D heat solver

The constants every run shares, built directly from `lunar.constants` so
the printed table can never disagree with what the solver actually uses.
These are *not* fitted — they are held fixed at their published values
while we retrieve K_d.

In [ ]:
from lunar import constants as c
import pandas as pd
rows = [
    # (parameter, symbol, value, unit, source)
    ('Surface thermal conductivity',    'K_s',        f'{c.K_SURFACE:.2e}',    'W m^-1 K^-1', 'Hayne 2017'),
    ('Deep thermal conductivity (publ.)', 'K_d (publ.)', f'{c.K_DEEP:.2e}',     'W m^-1 K^-1', 'Hayne 2017'),
    ('K(z) transition e-folding depth', 'H',          f'{c.H_PARAMETER}',      'm',           'Hayne 2017'),
    ('Radiative coefficient',           'chi',        f'{c.CHI_RADIATIVE}',    'dimensionless', 'Hayne 2017'),
    ('Radiative normalisation T',       'T_ref',      f'{c.T_REFERENCE}',      'K',           'Hayne 2017'),
    ('Surface bulk density',            'rho_s',      f'{c.RHO_SURFACE}',      'kg m^-3',     'Hayne 2017'),
    ('Deep bulk density',               'rho_d',      f'{c.RHO_DEEP}',         'kg m^-3',     'Hayne 2017'),
    ('Stefan-Boltzmann constant',       'sigma',      f'{c.SIGMA_SB:.4e}',        'W m^-2 K^-4', 'CODATA'),
]
df = pd.DataFrame(rows, columns=['Parameter', 'Symbol', 'Value', 'Unit', 'Source'])
df

### Site-specific inputs

The handful of numbers that differ between the two boreholes — latitude,
albedo, emissivity, the published basal heat flux Q_b, and the deepest
sensor depth. These come from `lunar/config.py` (`SITES`) and the cited
sources. The basal flux Q_b is held fixed during the K_d retrieval (the
paper's stated assumption).

In [ ]:
site_rows = [
    ('Latitude',        'lat',   '26.13 N',      '20.16 N',  'ALSEP gazetteer'),
    ('Longitude',       'lon',   '3.63 E',       '30.77 E',  'ALSEP gazetteer'),
    ('Bond albedo',     'A',     '0.131',        '0.137',    'Vasavada 2012'),
    ('Emissivity',      'eps',   '0.95',         '0.95',     'Bandfield 2015'),
    ('Basal heat flux', 'Q_b',   '21 mW m^-2',   '15 mW m^-2', 'Langseth 1976, Nagihara 2018'),
    ('Deepest sensor',  'z_max', '139 cm',       '234 cm',   'Nagihara 2018'),
]
df_site = pd.DataFrame(site_rows, columns=['Parameter', 'Symbol', 'Apollo 15', 'Apollo 17', 'Source'])
df_site